In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

from sklearn.metrics import roc_auc_score, mean_absolute_error
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder

In [27]:
root_path = "/home/stefan/ioai-prep/kits/unirea"

seed = 42

# Data

In [28]:
train_df = pd.read_csv(f"{root_path}/train.csv")
test_df = pd.read_csv(f"{root_path}/test.csv")

In [29]:
test_df.head()

,id,an,materie,liceu,judet,procent_reusita,numar_candidati,preferinta_materie
0,120760,2023,"ANATOMIE SI FIZIOLOGIE UMANA, GENETICA SI ECOL...","LICEUL TEORETIC ""DOSITEI OBRADOVICI"" TIMISOARA",Timiș,100.0,3,13.6
1,120761,2023,"ANATOMIE SI FIZIOLOGIE UMANA, GENETICA SI ECOL...","LICEUL TEHNOLOGIC ""PETRE IONESCU MUSCEL"" DOMNESTI",Argeș,100.0,1,2.1
2,120762,2023,"ANATOMIE SI FIZIOLOGIE UMANA, GENETICA SI ECOL...","LICEUL TEORETIC ""CONSTANTIN SERBAN"" ALESD",Bihor,100.0,28,46.7
3,120763,2023,"ANATOMIE SI FIZIOLOGIE UMANA, GENETICA SI ECOL...","COLEGIUL NATIONAL ""GEORGE BARITIU"" CLUJ-NAPOCA",Cluj,100.0,4,4.4
4,120764,2023,"ANATOMIE SI FIZIOLOGIE UMANA, GENETICA SI ECOL...",LICEUL GERMAN SEBES,Alba,100.0,1,7.1


# Subtask 1

In [30]:
key = 'COLEGIUL NATIONAL "UNIREA" FOCSANI'

pref_list = test_df[(test_df["an"] == 2023) & (test_df["liceu"] == key)].groupby(by="materie")[
    "preferinta_materie"
].mean().sort_values(ascending=False)
pref_list

materie
GENERAL                                                     100.0
LIMBA ROMANA                                                100.0
MATEMATICA MATE-INFO                                         54.2
ISTORIE                                                      31.6
FIZICA TEO                                                   27.4
LOGICA, ARGUMENTARE SI COMUNICARE                            17.5
INFORMATICA MI C-C++                                         16.5
MATEMATICA ST-NAT                                            14.2
GEOGRAFIE                                                    11.8
ANATOMIE SI FIZIOLOGIE UMANA, GENETICA SI ECOLOGIE UMANA     10.8
CHIMIE ORGANICA TEO NIVEL I-II                                7.5
BIOLOGIE VEGETALA SI ANIMALA                                  5.7
PSIHOLOGIE                                                    1.4
ECONOMIE                                                      0.5
FILOSOFIE                                                     0.5
IN

In [31]:
subtask1 = [pref_list.drop(["GENERAL", "LIMBA ROMANA"]).index[0]]

# Subtask 2

In [32]:
ls = train_df[train_df["materie"] == "INFORMATICA MI C-C++"].groupby(by="an")["preferinta_materie"].mean().sort_values(ascending=False)
ls

an
2022    10.880889
2018    10.459427
2021    10.243820
2017     9.747470
2019     9.695205
2020     9.590969
2016     8.761097
2015     8.705854
2014     8.345187
Name: preferinta_materie, dtype: float64

In [33]:
subtask2 = [ls.index[0]]

# Subtask 3

predict mean grade, for a given school and subject

In [34]:
def prep_data(train_df, test_df, label="medie"):
    # Target extraction
    y_train = train_df[label].copy()
    if label == "anomalie":
        y_train = y_train.astype(int)

    # Internal combination for leak-free feature engineering
    df = pd.concat([train_df, test_df], ignore_index=True).sort_values("an")

    # 1. Liceu-Materie history
    g = df.groupby(["liceu", "materie"])["medie"]
    df["prev_medie"] = g.shift(1)
    df["hist_medie"] = g.transform(lambda x: x.expanding().mean().shift(1))
    df["hist_std"] = g.transform(lambda x: x.expanding().std().shift(1))

    # 2. Liceu and Subject trends
    for col in ["judet", "materie"]:
        avg = df.groupby(["an", col])["medie"].mean().reset_index()
        avg.columns = ["an", col, f"prev_{col}_avg"]
        avg["an"] += 1
        df = df.merge(avg, on=["an", col], how="left")

    # Drop metadata and targets
    df = df.drop(columns=["id", "medie", "anomalie"], errors="ignore")

    # Impute and Encode
    for col in df.columns:
        if df[col].dtype == "str":
            df[col] = df[col].fillna(
                df[col].mode()
            )
        else:
            df[col] = df[col].fillna(df[col].mean())

    le = LabelEncoder()
    for col in ["liceu", "materie", "judet"]:
        df[col] = le.fit_transform(df[col].astype(str))

    # Split back to X_train and X_test
    X_train = df.iloc[: len(train_df)]
    X_test = df.iloc[len(train_df) :]

    return X_train, y_train, X_test

In [35]:
X, y, X_test_real = prep_data(train_df, test_df, label="medie")

In [36]:
selector = X["an"] < 2021
X_train, X_test, y_train, y_test = X[selector], X[~selector], y[selector], y[~selector]

In [37]:
def evaluate_reg(model):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return mean_absolute_error(y_test, preds)

In [38]:
reg_gb = HistGradientBoostingRegressor(
    loss="absolute_error",
    random_state=seed,
)

evaluate_reg(reg_gb)

0.4699349170631434

In [39]:
clf = reg_gb

In [40]:
subtask3 = clf.predict(X_test_real)

# Subtask 4

predict if a sample is an anomaly or not

In [41]:
X, y, X_test_real = prep_data(train_df, test_df, label="anomalie")
X_train, X_test, y_train, y_test = X[selector], X[~selector], y[selector], y[~selector]

In [42]:
def evaluate_cls(model):
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_test)[:, 1]
    mae = roc_auc_score(y_test, preds)
    return mae

In [43]:
gbc = HistGradientBoostingClassifier(random_state=seed)

evaluate_cls(gbc)

0.9682879909989511

In [44]:
model = gbc

In [45]:
subtask4 = model.predict_proba(X_test_real)[:, 1]

# Submission

In [46]:
test_df.shape, subtask4.shape

((13017, 8), (13017,))

In [ ]:
def build_subtask(sid, ans):
    if len(ans) == 1:
        return pd.DataFrame({
            "id": sid,
            "subtaskID": sid,
            "answer": ans
        })
    return pd.DataFrame({
            "id": test_df["id"],
            "subtaskID": sid,
            "answer": ans
        })

subtasks = [
    (1, subtask1),
    (2, subtask2),
    (3, subtask3),
    (4, subtask4),
]

submission_df = pd.concat([build_subtask(sid, ans) for sid, ans in subtasks])

In [48]:
submission_df.head()

,id,subtaskID,answer
0,1,1,MATEMATICA MATE-INFO
0,2,2,2022
0,120760,3,8.227232
1,120761,3,7.292038
2,120762,3,9.138775


In [49]:
submission_df.to_csv(f"{root_path}/submission.csv", index=False)